## Demonstration of 1D and 1D time dilated convolutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fpschill/cas_main_dl_2026/blob/main/09_1DConv.ipynb)

In this notebook you will use 1D causal convolutions to predict time series. You learn how 1D convolutions with causal padding work and see that dilated causal convolutions are able to capture long-range dependencies.

**Dataset:** We will use an artificial dataset with some long-range dependencies. We generate 1000 time series with the length of 128 time steps that all follow the same pattern: a fast changing sine wave where the amplitude is modulated by another sine wave with lower frequency. To make it a bit more challenging, we add some noise at each time step of the waves. Our goal is to first predict the next 10 time steps and then we want to predict for even longer time steps in the future.

**Content:**
* create an artificial dataset with some long-range dependencies
* visualize a sample time series
* use keras to train a CNN with 1D causal convolutions
* predict the next 10 time steps of the time series
* predict the next 80 time steps of the time series
* use keras to train a CNN with 1D dilated causal convolutions (which are able to deal with long-range dependencies)
* predict the next 80 time steps of the time series with 1D dilated causal convolutions

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"  # must be set BEFORE keras is imported
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('default')

import keras
from keras.models import Sequential
from keras.layers import Input, Conv1D, Dense, Lambda

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())

### Simulate some data

We will generate the time series ourselves. So first let us visualize the generating process. We multiply a fast sine wave with a slower sine wave and add a bit of random noise.

In [ ]:
# process: fast sine wave * slow sine wave + random noise
start = np.random.uniform(0, 2*np.pi) # Random start point
s=128+10
# only fast oscillations
plt.figure(figsize=(13,5))
plt.plot(np.sin(start + np.linspace(0, 20*np.pi, s)))
plt.show()
# only slow oscillations
plt.figure(figsize=(13,5))
plt.plot(np.sin(start + np.linspace(0, np.pi, s)))
plt.show()
# multiply fast with slow
plt.figure(figsize=(13,5))
plt.plot(np.sin(start + np.linspace(0, 20*np.pi, s))*np.sin(start + np.linspace(0, np.pi, s)))
plt.show()
# add noise
plt.figure(figsize=(13,5))
plt.plot(np.sin(start + np.linspace(0, 20*np.pi, s))*
         np.sin(start + np.linspace(0, np.pi, s))+
         np.random.normal(0,0.1,s))
plt.show()

In the next cell we generate the train and validation data (in total 1000 time series: 800 for training, 200 for validation). The goal is to learn from the past of the time series and predict the next 10 steps and later even more than "only" 10 steps. For the long-term predictions, we also keep the true values of the next 80 steps (`Y_long`). They are not used for training, only to check the predictions.

In [ ]:
np.random.seed(1) # Fixing the seed, so data generation is always the same
seq_length = 128  # Sequence length used for training
look_ahead =  10  # The number of data points the model should predict
look_ahead_long = 80  # The number of true future data points we keep to check the long-term predictions


def gen_data(size=1000, noise=0.1, seq_length=128, look_ahead=10, look_ahead_long=80): # We create 1000 observations of the process
    s = seq_length + look_ahead  # length of the time series used for training
    # time axis: t goes from 0 to 1 over the first s steps (like np.linspace(0, 1, s)) and continues
    # for the look_ahead_long true future values
    t = np.arange(seq_length + look_ahead_long) / (s - 1)
    d = np.zeros((size, len(t), 1))
    for i in range(size):
        start = np.random.uniform(0, 2*np.pi) # Random start point
        d[i,:,0] = (np.sin(start + 20*np.pi*t) *
                    np.sin(start + np.pi*t) +
                    np.random.normal(0,noise,len(t)))
    return d[:,0:seq_length], d[:,seq_length:s], d[:,seq_length:]


X, Y, Y_long = gen_data()

# lets plot the first two series
for i in range(2):
    plt.figure(figsize=(13,5))
    plt.plot(range(0, seq_length),X[i,:,0],'b-')
    plt.plot(range(seq_length, seq_length + look_ahead),Y[i,:,0],color='orange')

plt.show()
print('The training data X is the blue line and we want to forecast the next 10 steps Y, the orange line.')

In [ ]:
print(X.shape)
print(Y.shape)
print(Y_long.shape)

### 1D Convolution without dilation rate

Here we define a neural network with 1D convolutions and "causal" padding, in a later step we will also use a dilation rate.

With causal padding, the output at time step t only depends on the inputs up to time step t. The network outputs one value per time step, and with the `Lambda` layer we keep only the last 10 of them: these are the predictions for the next 10 time steps.

In [ ]:
def last_steps(x, n):
    # keep only the last n time steps
    return x[:, -n:, :]

model_1Dconv = Sequential()
ks = 5
model_1Dconv.add(Input(shape=(seq_length, 1)))
model_1Dconv.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal'))
model_1Dconv.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal'))
model_1Dconv.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal'))
model_1Dconv.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal'))
model_1Dconv.add(Dense(1))
model_1Dconv.add(Lambda(last_steps, arguments={'n': look_ahead}))


model_1Dconv.compile(optimizer='adam', loss='mean_squared_error')
model_1Dconv.summary()

In [ ]:
history = model_1Dconv.fit(X[0:800], Y[0:800],
                    epochs=50,
                    batch_size=128,
                    validation_data=(X[800:1000], Y[800:1000]),
                    verbose=1)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error Loss')
plt.title('Loss Over Time')
plt.legend(['Train','Valid'])
plt.show()

Now we want to use the trained model to predict the next 10 steps, which is what the model was trained on.

In addition, we want to predict for longer than just 10 steps: we predict the next 10 steps, take the predictions as new "true" observations and feed these values into the model again. When we do that, we can predict for any length we want. The function `forecast` in the next cell does this, and `plot_forecast` plots the result. Then we predict the next 10 and 80 steps.

The green curve represents the prediction, while the orange curve shows the true values.

In [ ]:
def forecast(model, x, steps):
    # predict steps * look_ahead time steps after the time series x (shape (seq_length, 1)):
    # predict the next look_ahead steps, append them to the time series and use them
    # as new "observations" to predict the following look_ahead steps, and so on
    data = x.copy()
    pred = []
    for _ in range(steps):
        res = model.predict(data[None, -seq_length:], verbose=0)[0]  # shape (look_ahead, 1)
        pred.append(res[:, 0])
        data = np.concatenate([data, res])
    return np.concatenate(pred)


def plot_forecast(model, i, steps):
    # plot the time series nr i, the forecast (green) and the true values (orange)
    pred = forecast(model, X[i], steps)
    plt.figure(figsize=(13,5))
    plt.plot(range(0, seq_length), X[i,:,0], color='blue')
    plt.plot(range(seq_length, seq_length + len(pred)), pred, color='green')
    plt.plot(range(seq_length, seq_length + len(pred)), Y_long[i,:len(pred),0], color='orange')
    plt.axvline(x=seq_length)
    plt.title(f"predict {len(pred)} steps ahead")
    plt.show()

In [ ]:
i = 950  # time series nr 950 (from the validation data)
plot_forecast(model_1Dconv, i, steps=1)  # predict 10 steps
plot_forecast(model_1Dconv, i, steps=8)  # predict 80 steps

### 1D Convolution with dilation rate

Here we define a neural network with 1D convolutions and "causal" padding, this time with dilation rate, so we are able to look back longer in time.

#### 🔧 **YOUR TASK:**
Add the `dilation_rate` argument to the convolutional layers (e.g. 1, 2, 4, 8).

In [ ]:
model_1Dconv_w_d = Sequential()
##################################################
###### your code here#############################


###### end of your code ##########################
##################################################

# stop here if the layers have not been added yet
if len(model_1Dconv_w_d.layers) == 0:
    raise RuntimeError("The network model_1Dconv_w_d has no layers yet: add them in this cell (see the hint below), then run the cell again.")

<details>
  <summary>💡 Click here for a hint:</summary>

Use the same layers as in the network without dilation, and add the argument `dilation_rate` to the 2nd, 3rd and 4th convolutional layer, with the dilation rates 2, 4 and 8:

```python
model_1Dconv_w_d.add(Input(shape=(seq_length, 1)))
model_1Dconv_w_d.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal'))
model_1Dconv_w_d.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal', dilation_rate=2))
model_1Dconv_w_d.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal', dilation_rate=4))
model_1Dconv_w_d.add(Conv1D(filters=32, activation="relu", kernel_size=ks, padding='causal', dilation_rate=8))
model_1Dconv_w_d.add(Dense(1))
model_1Dconv_w_d.add(Lambda(last_steps, arguments={'n': look_ahead}))
```
</details>

In [ ]:
model_1Dconv_w_d.compile(optimizer='adam', loss='mean_squared_error')
model_1Dconv_w_d.summary()

In [ ]:
history = model_1Dconv_w_d.fit(X[0:800], Y[0:800],
                    epochs=50,
                    batch_size=128,
                    validation_data=(X[800:1000], Y[800:1000]),
                    verbose=1)

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error Loss')
plt.title('Loss Over Time')
plt.legend(['Train','Valid'])
plt.show()

In [ ]:
i = 950  # time series nr 950 (from the validation data)
plot_forecast(model_1Dconv_w_d, i, steps=1)  # predict 10 steps
plot_forecast(model_1Dconv_w_d, i, steps=8)  # predict 80 steps

### 🔧 **YOUR TASK:**

Compare the long time predictions of the 1D convolutional neural network with and without dilation.
- What do you observe and how do you explain it?

<details>
  <summary>🔑 Click here to View Answers:</summary>

Compare the predictions (green) with the true values (orange). Both networks predict the next 10 steps well. In the long-term prediction, the network without dilation continues the fast oscillation, but its amplitude does not follow the true values (where the slow sine wave lets the amplitude shrink and grow again), and its mean can drift away from 0, a trend that does not exist in the data. The network with dilation follows the true values much better, including the slow change of the amplitude.

The explanation is the **receptive field**, i.e. how many past time steps a prediction can depend on. A causal convolution with kernel size 5 and dilation rate d looks (5-1)·d time steps further into the past. Without dilation, the 4 layers see 1 + 4·4 = 17 time steps, which is only about one period of the fast sine wave (about 14 time steps). From such a short window, the network cannot know at which phase of the slow sine wave it is (whether the amplitude is growing or shrinking), and the errors add up when the predictions are fed back into the model. With the dilation rates 1, 2, 4, 8, the receptive field is 1 + 4·(1+2+4+8) = 61 time steps, i.e. more than four periods of the fast sine wave. So the network can see how the amplitude changes and continue this long-range pattern, with exactly the same number of weights.
</details>